<a href="https://colab.research.google.com/github/Kongbeng-21/SuperAi/blob/main/601088_%E0%B8%81%E0%B8%A4%E0%B8%95%E0%B8%B4%E0%B8%98%E0%B8%B5_%E0%B8%89%E0%B8%B2%E0%B8%A2%E0%B9%81%E0%B8%AA%E0%B8%87.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

super_ai_engineer_season_6_ocr_2569_path = kagglehub.competition_download('super-ai-engineer-season-6-ocr-2569')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Setup

In [ ]:
!pip install google-generativeai pillow -q --no-deps

In [ ]:
from kaggle_secrets import UserSecretsClient
GEMINI_API_KEY = UserSecretsClient().get_secret("GMN")

# Import and Config

In [ ]:
from google import genai
import pandas as pd
import json
import re
import os
import time
from pathlib import Path
from collections import defaultdict
from PIL import Image

# single key
client = genai.Client(api_key=GEMINI_API_KEY)

# Paths
OUTPUT_PATH = "/kaggle/working/submission.csv"
CHECKPOINT_PATH = "/kaggle/working/checkpoint.json"
SUBMISSION_TEMPLATE = Path("/kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569/final_data/submission_template_v3.csv")
IMAGE_DIR = Path("/kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569/final_data/images")
MODEL_NAME = "gemini-3.1-flash-lite-preview"
DELAY_BETWEEN_CALLS = 8.0
MAX_RETRIES = 3
PARTY_BATCH_SIZE = 20

# Load template จัดกลุ่มภาพ

In [ ]:
def load_template(path):
    df = pd.read_csv(path)
    df.columns = ['id', 'party_name', 'votes']
    df['doc_id'] = df['id'].str.rsplit('_', n=1).str[0]
    print(f"✓ Template: {len(df)} rows, {df['doc_id'].nunique()} documents")
    return df

def group_pages_by_doc(image_dir):
    """จัดกลุ่ม PNG ตาม document id — scan จาก IMAGE_DIR เท่านั้น"""
    all_pngs = list(image_dir.rglob("*.png")) + list(image_dir.rglob("*.PNG"))

    doc_pages = defaultdict(list)
    for img_path in all_pngs:
        fname = img_path.stem
        doc_id = re.sub(r'_page\d+$', '', fname)
        doc_pages[doc_id].append(img_path)

    for doc_id in doc_pages:
        doc_pages[doc_id].sort(
            key=lambda p: int(re.search(r'page(\d+)', p.stem).group(1))
            if re.search(r'page(\d+)', p.stem) else 0
        )

    print(f"✓ พบ {len(doc_pages)} documents, {sum(len(v) for v in doc_pages.values())} หน้า")
    return doc_pages


# Prompt


In [ ]:
SYSTEM_PROMPT = """คุณคือระบบ OCR เชี่ยวชาญเอกสารผลเลือกตั้งไทย (แบบ สส.6/1)

กฎสำคัญ:
- ตอบเป็น JSON เท่านั้น ห้ามมี markdown หรือคำอธิบาย
- คะแนนในเอกสารอาจอยู่ในรูปแบบต่างๆ ให้แปลงเป็นตัวเลขอารบิกเสมอ:
  * เลขไทย เช่น ๑๒๓๔ → 1234
  * ตัวอักษรไทย เช่น "สี่หมื่นเก้าพันแปดร้อยสี่" → 49804
  * เลขอารบิกปกติ เช่น 1,234 → 1234
- คะแนนอยู่ในคอลัมน์ "ได้คะแนน" หรือ "คะแนน" ในตาราง
- ดูชื่อพรรคจากคอลัมน์ "สังกัดพรรคการเมือง"
- ถ้ามีการขีดฆ่าแล้วเขียนทับ ให้ใช้ค่าใหม่เท่านั้น
- ถ้าไม่พบพรรคในเอกสาร ให้ใส่ "0"
- votes ต้องเป็น string ตัวเลขล้วน เช่น "49804" ไม่ใช่ "49,804"
"""

def build_prompt(party_names: list) -> str:
    party_list = "\n".join(f"- {p}" for p in party_names)
    return f"""{SYSTEM_PROMPT}

จากภาพเอกสารผลเลือกตั้งที่ให้มา ให้หาคะแนนเสียงของแต่ละพรรคต่อไปนี้:

{party_list}

ตอบในรูปแบบ JSON เท่านั้น:
{{
  "ชื่อพรรค1": "คะแนน",
  "ชื่อพรรค2": "คะแนน",
  ...
}}

- ใช้ชื่อพรรคตามรายการด้านบนเป็น key ทุกตัว
- ถ้าไม่พบพรรคในเอกสาร ให้ใส่ "0"
- ดูข้อมูลจากทุกหน้าที่ให้มา
"""

def call_gemini_with_retry(images, prompt):
    for attempt in range(MAX_RETRIES):
        try:
            c = genai.Client(api_key=GEMINI_API_KEY)
            response = c.models.generate_content(
                model=MODEL_NAME,
                contents=[prompt] + images
            )
            raw = response.text.strip()
            clean = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw, flags=re.MULTILINE).strip()
            return json.loads(clean)
        except json.JSONDecodeError:
            print(f"  [WARN] JSON error รอบ {attempt+1}: {raw[:120]}")
            time.sleep(3)
        except Exception as e:
            err_str = str(e)
            if "429" in err_str or "quota" in err_str.lower():
                wait = 120 * (attempt + 1)
                print(f"  [RATE LIMIT] รอ {wait} วิ...")
                time.sleep(wait)
            elif "503" in err_str or "unavailable" in err_str.lower():
                print(f"  [503] model busy รอ 30 วิ...")
                time.sleep(30)
            else:
                print(f"  [ERROR] {e}")
                time.sleep(5)
    return None

def extract_votes(doc_id, page_paths, party_names):
    """ดึงคะแนนจากภาพ โดยแบ่ง party เป็น batch เพื่อความแม่นยำ"""
    images = []
    for p in page_paths:
        try:
            images.append(Image.open(p))
        except Exception as e:
            print(f"  [WARN] เปิดภาพไม่ได้: {e}")

    if not images:
        return {p: "0" for p in party_names}

    result = {}
    batches = [party_names[i:i+PARTY_BATCH_SIZE] for i in range(0, len(party_names), PARTY_BATCH_SIZE)]

    for batch_idx, batch in enumerate(batches):
        if len(batches) > 1:
            print(f"    batch {batch_idx+1}/{len(batches)} ({len(batch)} พรรค)")

        prompt = build_prompt(batch)
        batch_result = call_gemini_with_retry(images, prompt)

        if batch_result is None:
            # ล้มเหลวทุก retry — fallback เฉพาะ batch นี้
            print(f"    [FAIL] batch {batch_idx+1} ล้มเหลวทุก retry → ใส่ 0")
            for p in batch:
                result[p] = "0"
        else:
            for p in batch:
                raw_val = batch_result.get(p, "0")
                result[p] = str(raw_val).strip()

        if batch_idx < len(batches) - 1:
            time.sleep(DELAY_BETWEEN_CALLS)

    return result


# Post processing

In [ ]:
def normalize_votes(raw) -> str:
    """แปลงค่าให้เป็น string ตัวเลขอารบิกล้วน"""
    s = str(raw).strip()

    # แปลงเลขไทย
    s = s.translate(str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789'))

    # เอาเฉพาะตัวเลข
    s = re.sub(r'[^\d]', '', s)

    # ตัด leading zeros
    s = s.lstrip('0') or '0'

    return s if s else "0"

# Pipeline

In [ ]:
def load_checkpoint():
    """โหลด checkpoint ถ้ามี"""
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH) as f:
            data = json.load(f)
        print(f"✓ โหลด checkpoint: {len(data)} rows บันทึกแล้ว")
        return data
    return {}

def save_checkpoint(results):
    """บันทึก checkpoint ทุก document"""
    with open(CHECKPOINT_PATH, 'w') as f:
        json.dump(results, f, ensure_ascii=False)

def save_csv(results, template_df):
    all_ids = template_df['id'].tolist()
    submission = pd.DataFrame({'id': all_ids})
    submission['votes'] = submission['id'].map(results).fillna('0')
    submission.to_csv(OUTPUT_PATH, index=False)
    return submission

def run_pipeline():
    template_df = load_template(SUBMISSION_TEMPLATE)
    doc_pages = group_pages_by_doc(IMAGE_DIR)
    results = load_checkpoint()
    already_done = set(results.keys())
    docs = list(template_df.groupby('doc_id'))
    total = len(docs)
    skipped = 0

    print(f"\nเริ่มประมวลผล {total} documents (ทำแล้ว {len(already_done)} docs)...\n")

    for i, (doc_id, doc_rows) in enumerate(docs):
        row_ids = doc_rows['id'].tolist()
        if all(rid in already_done for rid in row_ids):
            skipped += 1
            continue

        print(f"[{i+1}/{total}] {doc_id} ({len(doc_rows)} พรรค, {len(doc_pages.get(doc_id, []))} หน้า)")

        pages = doc_pages.get(doc_id, [])
        if not pages:
            print(f"    ไม่พบภาพ — ใส่ 0 ทั้งหมด")
            for _, row in doc_rows.iterrows():
                results[row['id']] = "0"
            save_checkpoint(results)
            continue

        party_names = doc_rows['party_name'].tolist()
        vote_map = extract_votes(doc_id, pages, party_names)

        matched = 0
        for _, row in doc_rows.iterrows():
            raw = vote_map.get(row['party_name'], "0")
            results[row['id']] = normalize_votes(raw)
            if results[row['id']] != "0":
                matched += 1

        print(f"  ✓ พบคะแนน {matched}/{len(party_names)} พรรค")

        save_checkpoint(results)

        # save CSV ทุก 10 docs
        if (i + 1) % 10 == 0:
            save_csv(results, template_df)
            print(f"บันทึก CSV ชั่วคราว ({i+1} docs)")

        time.sleep(DELAY_BETWEEN_CALLS)

    if skipped:
        print(f"\n(ข้ามไป {skipped} docs ที่ทำแล้ว)")

    submission = save_csv(results, template_df)
    print(f"\n✓ บันทึกแล้ว: {OUTPUT_PATH}")
    print(f"   {len(submission)} rows")
    print(f"   votes='0': {(submission['votes']=='0').sum()} rows ({(submission['votes']=='0').mean():.1%})")
    print(submission.head())
    return submission

# Run

In [ ]:
submission = run_pipeline()

In [ ]:
import shutil
shutil.copy('/kaggle/working/submission.csv', '/kaggle/working/submission_download.csv')

In [ ]:
import os
print(os.path.exists('/kaggle/working/checkpoint.json'))

In [ ]:
import json, pandas as pd

with open('/kaggle/working/checkpoint.json') as f:
    results = json.load(f)

template_df = pd.read_csv('/kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569/final_data/submission_template_v3.csv')
template_df.columns = ['id', 'party_name', 'votes']

submission = pd.DataFrame({'id': template_df['id']})
submission['votes'] = submission['id'].map(results).fillna('0')
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(f"✓ {len(submission)} rows")
print(f"✓ zeros: {(submission['votes']=='0').sum()} rows")